In [ ]:
%load_ext autoreload
%autoreload 2

# ebieot_gmm ALAE — evaluation

Compare trained **EBiEOT GMM** checkpoints against optional **FSBM** baselines on FFHQ latents.
Merged from legacy `ebieot_gmm_ALAE_age.ipynb` and `ebieot_gmm_ALAE_gen.ipynb`.

Set Papermill parameter `TRNSF` to `"age"` (ADULT→CHILDREN) or `"gen"` (WOMAN→MAN).
Run with cwd `EBiEOT/` or `notebooks/ALAE/`.

## 1. Imports

In [ ]:
import gc
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import os
import sys
from itertools import cycle, islice
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from hydra import compose, initialize_config_dir
from omegaconf import DictConfig, OmegaConf
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from torchvision.utils import make_grid, save_image
from tqdm import tqdm, trange

from src.utils.notebook_setup import ensure_alae_notebook_path

NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "ALAE"
ensure_alae_notebook_path(NOTEBOOK_DIR)

from src.utils.datasets.alae import get_latents
from src.utils.core.seed import set_seed
from src.utils.samplers.data import TensorSampler


def _resolve_cfg(
    run_dir: Path,
    ckpt: dict[str, Any],
    config_path: Path | None = None,
) -> DictConfig:
    if config_path is not None:
        config_path = Path(config_path).expanduser().resolve()
        if not config_path.is_file():
            raise FileNotFoundError(f"config_path not found: {config_path}")
        return OmegaConf.load(config_path)
    hydra_cfg = run_dir / ".hydra" / "config.yaml"
    if hydra_cfg.is_file():
        return OmegaConf.load(hydra_cfg)
    if isinstance(ckpt, dict) and "cfg" in ckpt:
        return OmegaConf.create(ckpt["cfg"])
    raise FileNotFoundError(
        f"No config at {hydra_cfg}, config_path, or checkpoint['cfg'] for {run_dir}"
    )


def load_gmm_from_run(
    run_dir: Path,
    device: torch.device,
    *,
    checkpoint_name: str = "checkpoint.pt",
    config_path: Path | None = None,
) -> tuple[torch.nn.Module, DictConfig]:
    run_dir = Path(run_dir).expanduser().resolve()
    ckpt_path = run_dir / checkpoint_name
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = _resolve_cfg(run_dir, ckpt, config_path=config_path)
    model = build_gmm_model(cfg, device)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model, cfg


def normalize_tensor(tensor: torch.Tensor) -> torch.Tensor:
    return (tensor / 2 + 0.5).clamp_(0, 1)


def to_uint8(normalized_tensor: torch.Tensor) -> torch.Tensor:
    return normalized_tensor.mul(255).add_(0.5).clamp_(0, 255).to(torch.uint8)


In [ ]:
device = torch.device(
    f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
)
device

In [ ]:
torch.set_default_device(device)
dtype = torch.float32
torch.set_default_dtype(dtype)

## 2. Parameters

| `TRNSF` | Direction | Hydra experiment |
| --- | --- | --- |
| `age` | ADULT → CHILDREN | `gmm_alae_adult_children` |
| `gen` | WOMAN → MAN | `gmm_alae_woman_man` |

Point `EBIEOT_RUN_DIR` at a Hydra output folder (`checkpoint.pt` + `.hydra/config.yaml`) or a notebook
`checkpoints/<EXP_NAME>/` folder with `D_<step>.pt` and optional `CONFIG_PATH` to the saved Hydra config.

In [ ]:
# Papermill: TRNSF selects age vs gender transfer preset.
TRNSF = "age"  # "age" | "gen"
FSBM_SEED = 2  # 0, 1, 2 — used when FSBM_CKPT_DIR is resolved automatically
OVERRIDES: list[str] = []

# EBiEOT checkpoint (leave empty to skip ebieot_gmm eval sections)
EBIEOT_RUN_DIR = ""
EBIEOT_CHECKPOINT = "D_10000.pt"
CONFIG_PATH = ""

# External FSBM project (../FSBM relative to EBiEOT); empty → auto-detect, skip if missing
FSBM_ROOT = ""
FSBM_CKPT_DIR = ""  # optional explicit run subdir under outputs/runs/ffhq_{TRNSF}/

EVAL_MODEL_STEP = 10000
METRICS_BATCH_SIZE = 128
METRICS_USE_CYCLE: bool | None = None  # None → preset default (age: cycle, gen: aligned)

TRNSF_PRESETS = {
    "age": {
        "experiment": "gmm_alae_adult_children",
        "input_data": "ADULT",
        "target_data": "CHILDREN",
        "paired_test_slice": slice(None, 2000),
        "fsbm_subdirs": {
            0: "2025.11.27/161712",
            1: "2025.12.02/103030",
            2: "2025.12.02/153518",
        },
        "fsbm_traj_direction": "fwd",
        "metrics_use_cycle": True,
    },
    "gen": {
        "experiment": "gmm_alae_woman_man",
        "input_data": "WOMAN",
        "target_data": "MAN",
        "paired_test_slice": slice(2000, 4000),
        "fsbm_subdirs": {
            0: "2025.11.27/162837",
            1: "2025.12.02/124004",
            2: "2025.12.02/174536",
        },
        "fsbm_traj_direction": "bwd",
        "metrics_use_cycle": False,
    },
}

if TRNSF not in TRNSF_PRESETS:
    raise ValueError(f"TRNSF must be one of {list(TRNSF_PRESETS)}; got {TRNSF!r}")

preset = TRNSF_PRESETS[TRNSF]
EXPERIMENT = preset["experiment"]
INPUT_DATA = preset["input_data"]
TARGET_DATA = preset["target_data"]
PAIRED_TEST_SLICE = preset["paired_test_slice"]
FSBM_TRAJ_DIRECTION = preset["fsbm_traj_direction"]
if METRICS_USE_CYCLE is None:
    METRICS_USE_CYCLE = bool(preset["metrics_use_cycle"])

CONF_DIR = os.path.abspath(str(REPO_ROOT / "conf"))
compose_overrides = [f"experiment={EXPERIMENT}", *OVERRIDES]
with initialize_config_dir(version_base=None, config_dir=CONF_DIR):
    cfg = compose(config_name="config", overrides=compose_overrides)

INPUT_DATA = str(cfg.dataset.input_data)
TARGET_DATA = str(cfg.dataset.target_data)
seed = int(cfg.seed) if cfg.get("seed") is not None else int(cfg.train.seed)
set_seed(seed)
print(f"TRNSF={TRNSF}: {INPUT_DATA} -> {TARGET_DATA}")
print(OmegaConf.to_yaml(cfg.train))


## 3. Data

In [ ]:
ds = cfg.dataset
data_root = REPO_ROOT / str(ds.data_root)

X_train, X_test = get_latents(INPUT_DATA, from_dir=str(data_root), dtype=dtype)
Y_train, Y_test = get_latents(TARGET_DATA, from_dir=str(data_root), dtype=dtype)

X_sampler = TensorSampler(X_train.to(dtype), device=str(device))
Y_sampler = TensorSampler(Y_train.to(dtype), device=str(device))

pairs_dir = data_root / "pairs" / f"{INPUT_DATA}->{TARGET_DATA}"
for name in ("X_train.pt", "Y_train.pt", "X_test.pt", "Y_test.pt"):
    if not (pairs_dir / name).is_file():
        raise FileNotFoundError(f"Missing paired data: {pairs_dir / name}")

X_paired_test_full = torch.load(
    pairs_dir / "X_test.pt", map_location=device, weights_only=True
).to(dtype)
Y_paired_test_full = torch.load(
    pairs_dir / "Y_test.pt", map_location=device, weights_only=True
).to(dtype)
X_paired_test = X_paired_test_full[PAIRED_TEST_SLICE]
Y_paired_test = Y_paired_test_full[PAIRED_TEST_SLICE]
print(f"Paired test: {X_paired_test.shape[0]} pairs (slice {PAIRED_TEST_SLICE})")

## 4. Load EBiEOT checkpoint

In [ ]:
model: torch.nn.Module | None = None
OUTPUT_PATH: Path | None = None

if EBIEOT_RUN_DIR:
    run_dir = Path(EBIEOT_RUN_DIR).expanduser()
    if not run_dir.is_absolute():
        run_dir = (REPO_ROOT / run_dir).resolve()
    ckpt_name = EBIEOT_CHECKPOINT or "checkpoint.pt"
    cfg_path = Path(CONFIG_PATH).expanduser() if CONFIG_PATH else None
    model, cfg = load_gmm_from_run(
        run_dir,
        device,
        checkpoint_name=ckpt_name,
        config_path=cfg_path,
    )
    OUTPUT_PATH = run_dir
    print(f"Loaded EBiEOT from {run_dir / ckpt_name}")
else:
    model = build_gmm_model(cfg, device)
    model.init_a_by_samples(Y_sampler.sample(int(cfg.ebieot.model.n_potentials)))
    gm = cfg.ebieot.model
    cost = cfg.ebieot.cost
    paired_cfg = cfg.train.optimizer.paired
    unpaired_cfg = cfg.train.optimizer.unpaired
    P_XY = min(int(ds.P_XY_paired), 2000)
    Q_X = min(int(ds.Q_X_unpaired), X_train.shape[0])
    R_Y = min(int(ds.R_Y_unpaired), Y_train.shape[0])
    exp_name = (
        f"EBiEOT-GMM-ALAE-{INPUT_DATA}-to-{TARGET_DATA}-"
        f"P{P_XY}_Q{Q_X}_R{R_Y}_"
        f"N{gm.n_potentials}_M{cost.m_potentials}_"
        f"lr_p{float(paired_cfg.lr):.0e}_lr_u{float(unpaired_cfg.lr):.0e}"
    )
    OUTPUT_PATH = REPO_ROOT / "checkpoints" / exp_name
    legacy_ckpt = OUTPUT_PATH / f"D_{EVAL_MODEL_STEP}.pt"
    if legacy_ckpt.is_file():
        model.load_state_dict(
            torch.load(legacy_ckpt, map_location=device, weights_only=True)
        )
        model.eval()
        print(f"Loaded legacy checkpoint {legacy_ckpt}")
    else:
        print(
            "No EBIEOT_RUN_DIR and no legacy checkpoint; set EBIEOT_RUN_DIR or train with ebieot_gmm_alae.ipynb"
        )


## 5. FSBM baseline (optional)

Requires the external [`FSBM`](../../FSBM) repo next to `EBiEOT/`. Skipped when the path is absent.

In [ ]:
fsbm_model = None

_fsbm_root = Path(FSBM_ROOT).expanduser() if FSBM_ROOT else REPO_ROOT.parent / "FSBM"
if not _fsbm_root.is_dir():
    print(f"FSBM not found at {_fsbm_root}; skipping FSBM sections.")
else:
    if str(_fsbm_root) not in sys.path:
        sys.path.insert(0, str(_fsbm_root))
    from fsbm.utils import restore_model

    if FSBM_CKPT_DIR:
        fsbm_run = Path(FSBM_CKPT_DIR).expanduser()
        if not fsbm_run.is_absolute():
            fsbm_run = _fsbm_root / fsbm_run
    else:
        subdir = preset["fsbm_subdirs"].get(FSBM_SEED)
        if subdir is None:
            raise ValueError(f"Unknown FSBM_SEED={FSBM_SEED} for TRNSF={TRNSF}")
        fsbm_run = _fsbm_root / "outputs" / "runs" / f"ffhq_{TRNSF}" / subdir

    fsbm_cfg_path = fsbm_run / ".hydra" / "config.yaml"
    fsbm_ckpt_path = fsbm_run / "checkpoints" / "last.ckpt"
    if fsbm_cfg_path.is_file() and fsbm_ckpt_path.is_file():
        fsbm_model, _ = restore_model(str(fsbm_ckpt_path), device=device)
        fsbm_model.eval()
        print(f"Loaded FSBM from {fsbm_run}")
    else:
        print(f"FSBM checkpoint missing under {fsbm_run}; skipping.")

## 6. ALAE decoder

Vendored ALAE stack under `notebooks/ALAE/ALAE/` (adds `alae_ffhq_inference` on `sys.path`).

In [ ]:
alae_dir = NOTEBOOK_DIR / "ALAE"
if alae_dir.is_dir() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

alae_model = None
try:
    from alae_ffhq_inference import decode, load_model

    alae_cfg = alae_dir / "configs" / "ffhq.yaml"
    alae_artifacts = alae_dir / "training_artifacts" / "ffhq"
    if alae_cfg.is_file():
        alae_model = load_model(
            str(alae_cfg),
            training_artifacts_dir=str(alae_artifacts),
        ).to(device).to(dtype)
        alae_model.eval()
        print("ALAE decoder loaded.")
    else:
        print(f"ALAE config not found at {alae_cfg}; image metrics/plots will be skipped.")
except ImportError as exc:
    print(f"ALAE decode unavailable ({exc}); image metrics/plots will be skipped.")

In [ ]:
if fsbm_model is not None and alae_model is not None:
    x_demo = X_paired_test[:10]
    output = fsbm_model.sample(
        x_demo,
        log_steps=20,
        nfe=1000,
        direction=FSBM_TRAJ_DIRECTION,
    )
    y_pred = output["xs"]
    decoded = [normalize_tensor(decode(alae_model, traj)) for traj in y_pred]
    num_show = min(5, y_pred.shape[0])
    T = y_pred.shape[1]
    fig, axes = plt.subplots(num_show, T, figsize=(2 * T, 2 * num_show))
    for s in range(num_show):
        for t in range(T):
            axes[s, t].imshow(decoded[s][t].permute(1, 2, 0).cpu().numpy())
            axes[s, t].axis("off")
            if s == 0:
                axes[s, t].set_title(f"t={t}")
    plt.tight_layout()
    out_png = NOTEBOOK_DIR / f"{INPUT_DATA}->{TARGET_DATA}_FSBM_{FSBM_TRAJ_DIRECTION}.png"
    plt.savefig(out_png, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_png}")

## 7. Metrics (FID / SSIM / LPIPS)

In [ ]:
def eval_model(
    net: torch.nn.Module,
    model_name: str,
    *,
    use_cycle: bool = METRICS_USE_CYCLE,
) -> tuple[float, float, float]:
    if alae_model is None:
        raise RuntimeError("ALAE decoder required for image metrics")

    loss_fid = FrechetInceptionDistance().to(device)
    loss_ssim = StructuralSimilarityIndexMeasure(data_range=(-1.0, 1.0)).to(device)
    loss_lpip = LearnedPerceptualImagePatchSimilarity(net_type="alex").to(device)

    net.to(device)
    net.eval()
    alae_model.eval()

    with torch.no_grad():
        batch_size = METRICS_BATCH_SIZE
        if use_cycle:
            num_samples = max(len(X_test), len(Y_test))
        else:
            num_samples = min(len(X_test), len(Y_test))
        print(
            f"{model_name}: X_test={len(X_test)}, Y_test={len(Y_test)}, "
            f"using {num_samples} samples (cycle={use_cycle})"
        )
        num_iters = (num_samples + batch_size - 1) // batch_size

        if use_cycle:
            x_cycle = cycle(X_test)
            y_cycle = cycle(Y_test)
            for _ in tqdm(range(num_iters), desc=model_name):
                sub_batch_x = torch.stack(list(islice(x_cycle, batch_size))).to(device)
                sub_batch_y = torch.stack(list(islice(y_cycle, batch_size))).to(device)
                if "FSBM" in model_name:
                    out = net.sample(
                        sub_batch_x, log_steps=20, nfe=1000, direction="fwd"
                    )
                    y_pred = out["xs"][:, -1, :]
                else:
                    y_pred = net(sub_batch_x)
                pred_img = normalize_tensor(decode(alae_model, y_pred))
                true_img = normalize_tensor(decode(alae_model, sub_batch_y))
                loss_fid.update(to_uint8(pred_img), real=False)
                loss_fid.update(to_uint8(true_img), real=True)
                loss_ssim.update(pred_img, true_img)
                loss_lpip.update(pred_img, true_img)
                del sub_batch_x, sub_batch_y, y_pred
                torch.cuda.empty_cache()
        else:
            for i in tqdm(range(num_iters), desc=model_name):
                start = i * batch_size
                end = min(start + batch_size, num_samples)
                sub_batch_x = X_test[start:end].to(device)
                sub_batch_y = Y_test[start:end].to(device)
                if "FSBM" in model_name:
                    out = net.sample(
                        sub_batch_x, log_steps=20, nfe=1000, direction="fwd"
                    )
                    y_pred = out["xs"][:, -1, :]
                else:
                    y_pred = net(sub_batch_x)
                pred_img = normalize_tensor(decode(alae_model, y_pred))
                true_img = normalize_tensor(decode(alae_model, sub_batch_y))
                loss_fid.update(to_uint8(pred_img), real=False)
                loss_fid.update(to_uint8(true_img), real=True)
                loss_ssim.update(pred_img, true_img)
                loss_lpip.update(pred_img, true_img)
                del sub_batch_x, sub_batch_y, y_pred, pred_img, true_img
                torch.cuda.empty_cache()

    fid = float(loss_fid.compute())
    ssim = float(loss_ssim.compute())
    lpip = float(loss_lpip.compute())

    if OUTPUT_PATH is not None:
        for name, val in (
            ("FID", fid),
            ("SSIM", ssim),
            ("LPIP", lpip),
        ):
            torch.save(val, OUTPUT_PATH / f"{name}_{model_name}_{EVAL_MODEL_STEP}.pt")
    return fid, ssim, lpip

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
if model is not None and alae_model is not None:
    fid, ssim, lpips = eval_model(model, f"ebieot_{TRNSF}_seed{seed}")
    print(f"EBiEOT — FID: {fid:.4f}, SSIM: {ssim:.4f}, LPIPS: {lpips:.4f}")

In [ ]:
if fsbm_model is not None and alae_model is not None:
    fid, ssim, lpips = eval_model(fsbm_model, f"FSBM_{TRNSF}_seed{FSBM_SEED}")
    print(f"FSBM — FID: {fid:.4f}, SSIM: {ssim:.4f}, LPIPS: {lpips:.4f}")

## 8. Qualitative comparison

In [ ]:
def save_row(x, y, model_preds, model_names, num_gen, out_dir, idx):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    row = [x, y]
    for m_idx, _ in enumerate(model_names):
        for g in range(num_gen):
            row.append(model_preds[m_idx][g])
    row_t = torch.stack(row, dim=0)
    grid = make_grid(row_t, nrow=row_t.shape[0])
    save_image(grid, out_dir / f"row_{idx:05d}.png")


def generate_all_rows_batched(
    X_eval: torch.Tensor,
    Y_eval: torch.Tensor,
    models: list[nn.Module],
    model_names: list[str],
    batch_size: int = 64,
    num_gen: int = 1,
    out_dir: str | Path = "generated_rows",
    indices: list[int] | None = None,
):
    if alae_model is None:
        raise RuntimeError("ALAE decoder required for row export")
    dev = next(models[0].parameters()).device
    if indices is not None:
        indices = sorted(indices)
        num_samples = len(indices)
    else:
        num_samples = min(len(X_eval), len(Y_eval))
    num_batches = (num_samples + batch_size - 1) // batch_size
    out_dir = Path(out_dir)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, num_samples)
        batch_ids = list(range(start, end)) if indices is None else indices[start:end]
        x = X_eval[batch_ids].to(dev)
        y = Y_eval[batch_ids].to(dev)
        x_dec = normalize_tensor(decode(alae_model, x))
        y_dec = normalize_tensor(decode(alae_model, y))
        batch_model_preds = []
        for m, mname in zip(models, model_names):
            preds_gens = []
            for _ in range(num_gen):
                with torch.no_grad():
                    if mname == "FSBM":
                        out = m.sample(x, log_steps=20, nfe=1000, direction="fwd")
                        y_pred = out["xs"][:, -1, :]
                    else:
                        y_pred = m(x)
                    preds_gens.append(normalize_tensor(decode(alae_model, y_pred)))
            batch_model_preds.append(torch.stack(preds_gens, dim=1))
        for local_i in trange(len(batch_ids), leave=False):
            real_idx = batch_ids[local_i]
            preds_i = [m[local_i] for m in batch_model_preds]
            save_row(
                x_dec[local_i],
                y_dec[local_i],
                preds_i,
                model_names,
                num_gen,
                out_dir,
                real_idx,
            )
        del x, y, x_dec, y_dec, batch_model_preds
        torch.cuda.empty_cache()

In [ ]:
plot_models: list[nn.Module] = []
plot_names: list[str] = []

if alae_model is not None and (model is not None or fsbm_model is not None):
    num_images = min(10, X_paired_test.shape[0])
    num_gen = 5
    if model is not None:
        plot_models.append(model)
        plot_names.append("Our")
    if fsbm_model is not None:
        plot_models.append(fsbm_model)
        plot_names.append("FSBM")

    x = X_paired_test[:num_images]
    y = Y_paired_test[:num_images]
    init_img = normalize_tensor(decode(alae_model, x))
    true_img = normalize_tensor(decode(alae_model, y))

    all_model_preds = []
    for m, mname in zip(plot_models, plot_names):
        model_preds = []
        for _ in range(num_gen):
            with torch.no_grad():
                if mname == "FSBM":
                    out = m.sample(x, log_steps=20, nfe=1000, direction="fwd")
                    y_pred = out["xs"][:, -1, :]
                else:
                    y_pred = m(x)
                model_preds.append(normalize_tensor(decode(alae_model, y_pred)))
        all_model_preds.append(torch.stack(model_preds, dim=1))

    init_img_np = init_img.cpu().permute(0, 2, 3, 1).numpy()
    true_img_np = true_img.cpu().permute(0, 2, 3, 1).numpy()
    all_model_preds_np = [p.cpu().permute(0, 1, 3, 4, 2).numpy() for p in all_model_preds]

    cols = 2 + num_gen * len(plot_models)
    fig, axes = plt.subplots(num_images, cols, figsize=(cols, num_images * 1.5), dpi=200)
    if num_images == 1:
        axes = axes[None, :]
    for i in range(num_images):
        axes[i, 0].imshow(init_img_np[i])
        axes[i, 0].set_title("Input" if i == 0 else "")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(true_img_np[i])
        axes[i, 1].set_title("Target" if i == 0 else "")
        axes[i, 1].axis("off")
        col_idx = 2
        for m_idx, model_preds in enumerate(all_model_preds_np):
            for g_idx in range(num_gen):
                axes[i, col_idx].imshow(model_preds[i, g_idx])
                if i == 0:
                    axes[i, col_idx].set_title(plot_names[m_idx])
                axes[i, col_idx].axis("off")
                col_idx += 1
    plt.tight_layout(pad=0.5)
    full_png = NOTEBOOK_DIR / f"{INPUT_DATA}->{TARGET_DATA}_full.png"
    plt.savefig(full_png, bbox_inches="tight")
    plt.close()
    print(f"Saved {full_png}")

In [ ]:
if alae_model is not None and plot_models:
    rows_dir = NOTEBOOK_DIR / f"{INPUT_DATA}->{TARGET_DATA}_rows"
    generate_all_rows_batched(
        X_paired_test,
        Y_paired_test,
        models=plot_models,
        model_names=plot_names,
        batch_size=64,
        num_gen=1,
        out_dir=rows_dir,
    )
    print(f"Wrote comparison rows to {rows_dir}")